# Chapter 4 — 내 업무 문서로 AI 챗봇 만들기 (실습 노트북)

이 노트북은 강의 슬라이드 4-1 ~ 4-5 의 내용을 **한 셀씩 직접 실행**하며 데이터가 어떻게 변하는지 눈으로 확인할 수 있도록 구성되어 있습니다.

## 학습 목표
- 사내 문서를 청킹·임베딩·검색하는 전체 파이프라인을 손에 잡히게 이해
- 매 단계의 데이터(원문 → 청크 → 벡터 → 검색 결과 → 답변)를 직접 출력해 확인
- RAG 없을 때 vs 있을 때 답변 차이를 같은 질문으로 비교
- 본인 PDF 한 부로 직접 챗봇 만들어보기 (마지막 섹션)

## 실행 환경
- **Colab 추천** (PyTorch · sentence-transformers 자동 설치)
- 또는 로컬 Python 3.10+
- OpenAI API 키 필요 (LLM 답변 생성용)

> 셀을 위에서부터 차례로 실행하세요. 각 셀의 `print` 출력이 그 단계의 핵심 데이터입니다.


---
## 4-1. 왜 AI 가 우리 회사 문서를 모르는가

상용 LLM (GPT, Claude 등) 은 공개 인터넷 데이터로 학습되었기 때문에 **사내 매뉴얼·계약서·규정**은 학습 내용에 들어가 있지 않습니다.

**RAG (Retrieval-Augmented Generation)** 의 한 줄 정의:

> "AI 에게 내 문서를 *실시간으로* 읽혀주고, 그 내용을 근거로 답하게 하는 방식."

이번 챕터에서는 RAG 의 전체 흐름을 5단계로 직접 만들어 봅니다.

```
① 문서 업로드 → ② 청킹 → ③ 임베딩 → ④ 검색 → ⑤ AI 답변
```


---
## 4-2. 문서 처리 환경 셋업

먼저 필요한 라이브러리를 설치합니다. Colab 이면 첫 셀 한 번만 실행하면 됩니다.

| 라이브러리 | 용도 |
|---|---|
| `sentence-transformers` | 임베딩 모델 (문서 → 벡터) |
| `openai` | LLM API 호출 (답변 생성) |
| `numpy` | 벡터 유사도 계산 |
| `pypdf` | (선택) PDF 텍스트 추출 |


In [ ]:
!pip install -q sentence-transformers openai numpy pypdf

**OpenAI API 키 입력.** 노트북 안에 키를 하드코딩하지 말고 `getpass` 로 안전하게 입력받습니다.

키 발급: https://platform.openai.com/api-keys

> 셀 실행 후 입력창에 키를 붙여넣고 엔터.


In [ ]:
import os, getpass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API 키를 입력하세요: ')

print('API 키 설정 완료 (앞 7자):', os.environ['OPENAI_API_KEY'][:7] + '...')

### 샘플 사내 매뉴얼

실습용으로 **가상의 사내 취업규칙** 일부를 준비했습니다. 실제 업무 문서로 바꿔 쓸 수도 있습니다 (이 노트북 맨 끝 섹션 참고).

이 텍스트가 우리의 "지식 소스" 가 됩니다.


In [ ]:
SAMPLE_DOC = """[사내 취업규칙 — 발췌]

제15조 (연차유급휴가)
입사 후 1년 이상 근속한 사원에게는 매년 15일의 연차유급휴가를 부여한다.
1년 미만 근속 사원은 1개월 개근 시 1일의 유급휴가가 발생한다.
연차 사용 시에는 최소 3일 전에 사내 시스템을 통해 신청해야 하며 팀장의 승인을 받아야 한다.

제22조 (출장)
국내 출장 신청은 출장 시작 5영업일 전, 해외 출장은 10영업일 전까지 신청해야 한다.
출장비는 일비 5만원, 교통비 실비, 숙박비는 직급별 한도 내 실비로 정산한다.
출장 종료 후 5영업일 이내에 영수증 첨부하여 정산 신청을 완료해야 한다.

제30조 (재택근무)
주 2회까지 재택근무가 가능하며, 전일 18시까지 사내 메신저로 팀장에게 신청한다.
재택근무 중에는 사내 시스템에 로그인 상태를 유지하고, 핵심 근무시간(10시~16시) 동안
즉시 응답이 가능해야 한다.

제35조 (경조사 휴가)
본인 결혼 5일, 자녀 결혼 1일, 배우자 사망 5일, 부모 사망 5일, 형제자매 사망 3일을 부여한다.
경조사 휴가는 사유 발생일로부터 30일 이내에 신청해야 한다.

제40조 (교육 지원)
직무 관련 외부 교육 수강 시 연 100만원 한도로 교육비를 지원한다.
교육 신청은 부서장 결재 후 인사팀에 신청서를 제출한다.
지원 받은 교육 수료 후 1년 이내 퇴사 시 교육비 일부를 환수할 수 있다.

제45조 (시간외 근무)
법정 근로시간을 초과하는 근무는 사전에 팀장 승인이 필요하다.
시간외 근무 수당은 통상임금의 1.5배로 지급된다.
월 시간외 근무는 52시간을 초과할 수 없다.
"""

print(f'문서 길이: {len(SAMPLE_DOC)} 자')
print(f'문서 앞 200 자 미리보기:')
print(SAMPLE_DOC[:200] + '...')

---
### 청킹 — 문서를 작은 조각으로 자르기

AI 가 한 번에 읽을 수 있는 양은 제한됩니다 (context window). 또 검색 정확도를 위해 **의미 단위** 로 잘게 나눠야 합니다.

#### 나쁜 청킹: 글자 수로 무지성 자르기

먼저 "200자씩 그냥 자르기" 를 해보고 어떤 문제가 있는지 직접 보겠습니다.


In [ ]:
def bad_chunk(text: str, size: int = 200):
    """단순 글자 수 단위로 자르기. 의미 구조 무시."""
    return [text[i:i+size] for i in range(0, len(text), size)]

bad_chunks = bad_chunk(SAMPLE_DOC, size=200)
print(f'나쁜 청크 개수: {len(bad_chunks)}\n')
for i, c in enumerate(bad_chunks[:3]):
    print(f'--- 청크 {i+1} ---')
    print(c)
    print()

**관찰:** 청크 경계가 문장 중간을 자릅니다. "팀장의 승인을" 처럼 의미가 끊김 → 검색이 잘못 매칭될 수 있고, AI 가 받아도 어색한 문장으로 인식.

#### 좋은 청킹: 단락 경계 우선

빈 줄 (`\n\n`) 로 단락을 먼저 나누고, 단락이 너무 길면 문장 종결 패턴으로 자릅니다.


In [ ]:
import re

def good_chunk(text: str, size: int = 300):
    """단락 경계 우선 + 문장 경계 폴백."""
    # 1) 단락 단위로 먼저 쪼개기
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n+", text) if p.strip()]
    chunks = []
    buf = ""
    for p in paragraphs:
        if len(buf) + len(p) <= size:
            buf = (buf + "\n\n" + p).strip() if buf else p
            continue
        if buf:
            chunks.append(buf)
            buf = ""
        # 단락이 size보다 크면 문장 경계로 더 자르기
        if len(p) > size:
            sentences = re.split(r"(?<=[.!?다요죠음됨함])\s+", p)
            sbuf = ""
            for s in sentences:
                if len(sbuf) + len(s) <= size:
                    sbuf = (sbuf + " " + s).strip() if sbuf else s
                else:
                    if sbuf:
                        chunks.append(sbuf)
                    sbuf = s
            if sbuf:
                chunks.append(sbuf)
        else:
            buf = p
    if buf:
        chunks.append(buf)
    return chunks

chunks = good_chunk(SAMPLE_DOC, size=300)
print(f'좋은 청크 개수: {len(chunks)}\n')
for i, c in enumerate(chunks):
    print(f'--- 청크 {i+1} ({len(c)}자) ---')
    print(c)
    print()

**관찰:** 각 청크가 "한 조항" 단위로 완결됩니다. 의미가 중간에 끊기지 않음 → 검색 시 정확도 ↑, AI 도 더 잘 해석.

> 💡 실제 서비스에서는 청크 크기는 500~1000자가 보통. 짧은 문서는 작게, 긴 문서는 크게.


---
## 4-3. 임베딩 — 의미를 숫자로 바꾸기

각 청크를 **고정 길이 벡터(좌표)** 로 변환합니다. 비슷한 의미의 텍스트는 좌표가 가까이 놓입니다.

- 사용 모델: `paraphrase-multilingual-MiniLM-L12-v2` (한국어 포함 50+ 언어, ~470MB)
- 출력 차원: **384**

> Colab 첫 실행 시 모델 다운로드에 30초~1분 소요.


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

print('임베더 로딩 중... (첫 실행 시 모델 다운로드)')
embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
print('완료.')

# 각 청크를 벡터로 변환
chunk_embeddings = embedder.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True,  # cosine 유사도 쓰려면 정규화
    show_progress_bar=False,
)
print(f'\n임베딩 행렬 shape: {chunk_embeddings.shape}')
print(f'(청크 {chunk_embeddings.shape[0]}개 × 차원 {chunk_embeddings.shape[1]})')

print(f'\n첫 청크의 벡터 앞 10차원:')
print(chunk_embeddings[0][:10])

### 의미 유사도 확인 — "가까운" 의미를 직접 측정

각 청크 벡터끼리의 **코사인 유사도** 를 계산해 봅니다. 0~1 범위 (1이 똑같음). 의미가 비슷할수록 1에 가까워야 합니다.


In [ ]:
# 모든 청크 쌍의 유사도 행렬
sim_matrix = chunk_embeddings @ chunk_embeddings.T
print('청크 간 유사도 행렬:')
print(np.round(sim_matrix, 2))
print()
print('해석: 대각선은 자기 자신 (1.0). 비-대각선 값이 클수록 의미가 가까움.')

**관찰:** 모든 청크가 같은 사내 규정이라 전체적으로 0.4~0.7 정도 유사도가 나옵니다. 완전 다른 도메인 텍스트라면 0에 가깝게 떨어집니다.

이번엔 외부 쿼리 임베딩 → 청크와 유사도 비교해 봅시다.


In [ ]:
queries = [
    "연차 며칠이야",
    "재택 어떻게 신청해",
    "출장 정산은 어떻게",
    "주식 시장 분석",  # 문서와 무관 → 유사도 낮아야 함
]

q_embs = embedder.encode(queries, convert_to_numpy=True, normalize_embeddings=True)

for q, qe in zip(queries, q_embs):
    sims = chunk_embeddings @ qe
    top_idx = sims.argmax()
    print(f'질문: "{q}"')
    print(f'  → 가장 가까운 청크 (#{top_idx+1}, 유사도={sims[top_idx]:.3f}):')
    print(f'    {chunks[top_idx][:80]}...')
    print()

**관찰:** 
- "연차 며칠이야" → 제15조 (연차) 청크가 1등으로 검색됨
- "재택 어떻게" → 제30조 (재택) 청크
- "주식 시장" → 유사도가 다른 것들보다 현저히 낮음 (문서와 무관)

**이게 의미 검색의 핵심.** 키워드가 정확히 매칭되지 않아도 의미가 비슷하면 찾아냅니다.


---
### 검색 함수 — top-k 청크 가져오기

같은 로직을 함수로 묶어둡니다. 이후 LLM 호출 때 재사용.


In [ ]:
def search(query: str, top_k: int = 3):
    """질문 → 임베딩 → 청크 임베딩과 코사인 유사도 → top_k 반환."""
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    sims = chunk_embeddings @ q_emb
    top_idx = np.argsort(-sims)[:top_k]
    return [
        {'rank': i+1, 'score': float(sims[idx]), 'chunk_idx': int(idx),
         'text': chunks[idx]}
        for i, idx in enumerate(top_idx)
    ]

# 테스트
results = search("출장 다녀와서 정산 어떻게 해", top_k=3)
for r in results:
    print(f'[{r["rank"]}] (청크 #{r["chunk_idx"]+1}, score={r["score"]:.3f})')
    print(f'    {r["text"][:120]}...')
    print()

---
## 4-4. AI 모델 연동 — RAG 있을 때 vs 없을 때

이제 검색 결과를 LLM 에게 컨텍스트로 전달하고 답변을 생성합니다.

같은 질문을:
1. **RAG 없이** (그냥 묻기) — 사내 규정 모름
2. **RAG 있이** (검색된 청크 + 질문) — 사내 규정 근거로 답변

으로 비교해 차이를 직접 봅니다.


In [ ]:
from openai import OpenAI

client = OpenAI()
MODEL = 'gpt-4o-mini'   # 빠르고 저렴. 더 정확하게는 'gpt-4o' 또는 'gpt-5-mini' 등

QUESTION = "우리 회사 연차는 며칠이고 어떻게 신청해?"
print(f'질문: {QUESTION}\n')

#### 비교 1: RAG 없이 그냥 묻기


In [ ]:
resp_no_rag = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'user', 'content': QUESTION},
    ],
)
print('═══ RAG 없이 ═══')
print(resp_no_rag.choices[0].message.content)

**관찰:** 일반론적 답변 ("일반적으로 근로기준법은..."). 우리 회사 구체 규정은 모름.

#### 비교 2: RAG 적용 — 검색된 청크를 컨텍스트로


In [ ]:
# 1) 질문으로 청크 검색
retrieved = search(QUESTION, top_k=3)

# 2) 검색 결과를 LLM 프롬프트에 컨텍스트로 추가
context = "\n\n".join(f'[{r["rank"]}] {r["text"]}' for r in retrieved)
system_prompt = (
    "당신은 사내 규정 전문 도우미입니다. 아래 [Context] 안의 내용만 근거로 답변하세요. "
    "Context 에 없는 내용은 절대 만들어내지 말고 '문서에서 확인되지 않습니다' 라고 답하세요. "
    "각 주장에는 [숫자] 형식으로 인용 번호를 붙이세요."
)
user_msg = f"[Context]\n{context}\n\n[질문] {QUESTION}"

print('═══ LLM 에 전달되는 프롬프트 (앞부분) ═══')
print(f'(system) {system_prompt[:100]}...')
print(f'(user)\n{user_msg[:400]}...')
print()

In [ ]:
resp_rag = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': user_msg},
    ],
)
print('═══ RAG 적용 답변 ═══')
print(resp_rag.choices[0].message.content)

**관찰:** 이번엔 **제15조 인용** + 구체적 일수 + 신청 절차까지. 사내 규정에 근거한 정확한 답변.

이게 RAG 의 본질입니다 — 같은 모델, 같은 질문, 단지 **컨텍스트 한 줄을 추가** 했더니 답변 품질이 완전히 달라짐.


---
### 프롬프트 설계 — 답변 품질의 또 다른 결정자

같은 검색 결과라도 **system prompt** 가 부실하면 모델이 환각을 합니다. 직접 비교해보겠습니다.


In [ ]:
BAD_PROMPT = "문서 보고 답해줘"
GOOD_PROMPT = (
    "아래 [Context] 만을 근거로 답하세요. "
    "Context 에 없는 내용은 절대 추측하지 말고 '문서에서 확인되지 않습니다' 라고 답하세요. "
    "각 주장에 [숫자] 인용을 붙이세요."
)

# 일부러 문서에 없는 내용을 묻기
HARD_Q = "퇴직금 산정 기준이 어떻게 돼?"   # 우리 SAMPLE_DOC 에는 없음

retrieved = search(HARD_Q, top_k=3)
context = "\n\n".join(f'[{r["rank"]}] {r["text"]}' for r in retrieved)
user_msg = f"[Context]\n{context}\n\n[질문] {HARD_Q}"

print(f'질문: {HARD_Q}')
print('(이 질문에 대한 정보는 우리 문서에 없습니다)\n')

for label, prompt in [('나쁜 프롬프트', BAD_PROMPT), ('좋은 프롬프트', GOOD_PROMPT)]:
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': prompt},
            {'role': 'user',   'content': user_msg},
        ],
    )
    print(f'═══ {label} ═══')
    print(r.choices[0].message.content)
    print()

**관찰:**
- 나쁜 프롬프트 → 모델이 추측해서 일반 근로기준법 내용을 지어낼 수 있음 (환각 위험)
- 좋은 프롬프트 → "문서에서 확인되지 않습니다" 로 정직하게 답함

> 좋은 RAG = (좋은 검색) + (강한 system prompt)


---
## 4-5. ask_question() — 통합 함수

지금까지 했던 단계 (검색 + 프롬프트 + LLM) 를 함수 하나로 묶습니다. 그러면 어떤 질문이든 한 줄로 답을 받을 수 있습니다.


In [ ]:
SYSTEM_PROMPT = (
    "당신은 사내 규정 전문 도우미입니다. 아래 [Context] 만을 근거로 한국어로 답변하세요. "
    "Context 에 없는 내용은 '문서에서 확인되지 않습니다' 라고만 답하고 추측하지 마세요. "
    "각 주장에 [숫자] 인용을 붙이세요."
)

def ask_question(question: str, top_k: int = 3, show_context: bool = False) -> str:
    """질문 → 검색 → 컨텍스트 + 질문을 LLM 에 전달 → 답변."""
    retrieved = search(question, top_k=top_k)
    context = "\n\n".join(f'[{r["rank"]}] {r["text"]}' for r in retrieved)

    if show_context:
        print('--- 검색된 컨텍스트 ---')
        print(context)
        print('--- 끝 ---\n')

    user_msg = f"[Context]\n{context}\n\n[질문] {question}"
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': user_msg},
        ],
    )
    return resp.choices[0].message.content

# 한 줄로 호출 가능
print(ask_question("재택근무 신청은 언제까지 해야 해?"))

### 다양한 질문 유형 테스트

슬라이드의 3가지 유형 (사실 / 절차 / 비교) 으로 챗봇이 어떻게 답하는지 확인.


In [ ]:
test_questions = [
    "연차는 며칠이야?",                         # 사실
    "출장 신청 어떻게 해?",                     # 절차
    "재택근무랑 출장 신청 기한이 어떻게 달라?",   # 비교 (멀티-홉)
    "주식 투자 조언 좀 해줘",                   # 문서 무관 → 거절해야 함
]

for q in test_questions:
    print(f'\n질문: {q}')
    print('-' * 60)
    print(ask_question(q))
    print()

**관찰:**
1. 사실 질문 → 정확한 일수 + 조항 인용
2. 절차 질문 → 신청 시점 + 정산 절차 모두 답함
3. 비교 질문 → 두 조항 모두 검색하고 차이 비교
4. 무관 질문 → "문서에서 확인되지 않습니다" 로 정직하게 답

이게 완성된 RAG 챗봇의 동작입니다.


---
## 4-5. Streamlit 으로 웹 챗봇 화면 만들기

이 노트북의 코드를 그대로 옮기고 `st.chat_input` / `st.chat_message` 로 감싸면 웹 챗봇이 됩니다.

```python
import streamlit as st

q = st.chat_input("질문하세요")
if q:
    with st.chat_message("user"):
        st.markdown(q)
    with st.chat_message("assistant"):
        answer = ask_question(q)
        st.markdown(answer)
```

전체 동작 예시는 본 챕터 참고 프로젝트(Personal RAG)에서 확인 가능합니다:
- 데모: https://personal-rag-real-lab.streamlit.app/
- 코드: https://github.com/REAL-KENTECH/Personal-RAG


---
## 심화. 실서비스 챗봇으로 가는 3가지 기법

지금까지 만든 RAG 는 "동작은 하는" 수준입니다. 실제 사내 챗봇으로 배포하면 다음 같은 상황에서 자주 실패합니다:

| 흔한 실패 | 원인 |
|---|---|
| "52시간 초과는 안 되나?" → 엉뚱한 청크 검색 | Dense 임베딩은 *숫자/고유어*에 약함 |
| top-5 검색 결과가 다 비슷한 청크 | 1차 검색만으로는 정밀도 부족 |
| 청크 크기 바꿨는데 진짜 좋아진 건지 모름 | 측정이 없음 = "느낌" 으로 튜닝 |

이 셋을 풀어주는 게 다음 세 가지입니다.

| 기법 | 한 줄 |
|---|---|
| **A. 하이브리드 검색** | Dense(의미) + BM25(키워드) + RRF 로 결합 |
| **B. 재정렬 (Reranker)** | 1차로 많이 뽑고, 더 정밀한 모델로 다시 정렬 |
| **C. 평가 (Recall@k)** | 정답 셋으로 측정해서 진짜 좋아졌는지 검증 |


### A. 하이브리드 검색 — Dense + BM25 + RRF

**Dense 임베딩 단점:** "52시간", "제30조", "OO-2025-001" 같은 **고유 숫자/식별자**는 의미 벡터에 잘 안 잡힙니다. 반면 BM25 는 단어 빈도/희소성 기반이라 이런 keyword 매칭에 강합니다.

**RRF (Reciprocal Rank Fusion):** 두 랭킹을 점수가 아니라 *순위* 로 합칩니다. 점수 스케일 다른 검색기를 섞을 때 가장 안전한 방법.

먼저 BM25 검색기를 만들어보겠습니다.


In [ ]:
!pip install -q rank_bm25

In [ ]:
from rank_bm25 import BM25Okapi
import re as _re

def tokenize_ko(text: str):
    """한국어 + 영어 + 숫자 혼합 토큰화. CJK 한 글자 단위로 자르면 짧은
    쿼리에서 매칭이 너무 좁아져서, 단어 단위(공백/구두점 split)로 시작."""
    text = (text or '').lower()
    # 단어 + CJK 연속체 + 숫자
    return _re.findall(r"[\w가-힯一-鿿]+", text)

# 청크 토큰화 후 BM25 인덱스 구축
tokenized_corpus = [tokenize_ko(c) for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)
print(f'BM25 코퍼스: 청크 {len(tokenized_corpus)}개, 첫 청크 토큰 수: {len(tokenized_corpus[0])}')
print('첫 청크 토큰 샘플:', tokenized_corpus[0][:10])

**Dense 가 약하지만 BM25 가 강한 쿼리를 직접 비교.** 

"시간외 근무는 월 52시간을 초과할 수 없다" 는 제45조에 있습니다. "52" 라는 숫자가 핵심인 쿼리를 두 방식으로 검색해서 차이를 봅니다.


In [ ]:
def dense_search_simple(query: str, top_k: int = 5):
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    sims = chunk_embeddings @ q_emb
    idx = np.argsort(-sims)[:top_k]
    return [(int(i), float(sims[i])) for i in idx]

def bm25_search(query: str, top_k: int = 5):
    scores = bm25.get_scores(tokenize_ko(query))
    idx = np.argsort(-scores)[:top_k]
    return [(int(i), float(scores[i])) for i in idx]

TRICKY_Q = "52시간 초과해서 일하면 어떻게 돼?"
print(f'질문: {TRICKY_Q}\n')

print('--- Dense (의미 검색) ---')
for i, s in dense_search_simple(TRICKY_Q, top_k=3):
    print(f'  청크#{i+1} (score={s:.3f}): {chunks[i][:60]}...')

print('\n--- BM25 (키워드 검색) ---')
for i, s in bm25_search(TRICKY_Q, top_k=3):
    print(f'  청크#{i+1} (score={s:.3f}): {chunks[i][:60]}...')

**관찰 포인트:** Dense 가 1등으로 시간외근무 조항을 못 잡거나, BM25 가 정확히 잡거나, 또는 둘이 다른 청크를 1등으로 추천하는 경우가 보일 거예요. 이런 *상보적* 패턴이 하이브리드 검색이 이기는 이유입니다.

이제 RRF 로 두 랭킹을 결합합니다.


In [ ]:
def rrf_fuse(rankings: list, k: int = 60):
    """rankings: [[(idx, score), ...], ...]  — 점수가 아닌 *순위*로 결합.
    k=60은 RRF 원논문(Cormack et al. 2009) 기본값."""
    scores = {}
    for ranking in rankings:
        for rank, (idx, _) in enumerate(ranking):
            scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: -x[1])

def hybrid_search(query: str, top_k: int = 5):
    dense = dense_search_simple(query, top_k=10)
    keyword = bm25_search(query, top_k=10)
    fused = rrf_fuse([dense, keyword])[:top_k]
    return fused  # [(idx, rrf_score), ...]

print(f'질문: {TRICKY_Q}\n')
print('--- 하이브리드 (Dense + BM25, RRF 결합) ---')
for i, s in hybrid_search(TRICKY_Q, top_k=3):
    print(f'  청크#{i+1} (rrf={s:.4f}): {chunks[i][:60]}...')

### B. 재정렬 — Cross-encoder 로 정밀도 한 번 더 올리기

**임베딩 모델 두 종류 차이:**

- **Bi-encoder** (지금까지 우리가 쓴 MiniLM): 쿼리와 청크를 *따로* 인코딩 후 벡터 유사도. **빠름 + coarse**.
- **Cross-encoder** (Reranker): 쿼리와 청크를 *함께* 모델에 넣어 두 텍스트 관련성을 직접 점수화. **느림 + precise**.

**전략:** Bi-encoder 로 top-20 빠르게 추리고 → Cross-encoder 로 top-5 정밀하게 다시 정렬.

> Colab 첫 실행 시 reranker 다운로드에 1-2분 (≈580MB).


In [ ]:
from sentence_transformers import CrossEncoder

print('Reranker 로딩 중...')
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3')
print('완료.')

def rerank(query: str, candidate_idx: list, top_k: int = 5):
    """candidate_idx: 1차 검색에서 나온 청크 인덱스 리스트.
    cross-encoder가 (query, chunk) 쌍을 직접 점수화."""
    pairs = [[query, chunks[i]] for i in candidate_idx]
    scores = reranker.predict(pairs, show_progress_bar=False)
    ranked = sorted(zip(candidate_idx, scores), key=lambda x: -x[1])
    return ranked[:top_k]

# 1차: 하이브리드 검색으로 top-10 후보
Q = "재택근무 신청 기한이 언제까지야?"
first_pass = hybrid_search(Q, top_k=10)
first_pass_idx = [i for i, _ in first_pass]

# 2차: 재정렬로 top-3
reranked = rerank(Q, first_pass_idx, top_k=3)

print(f'질문: {Q}\n')
print('--- 1차 (Hybrid, top-3) ---')
for i, s in first_pass[:3]:
    print(f'  청크#{i+1} (rrf={s:.4f}): {chunks[i][:60]}...')
print('\n--- 2차 (Rerank, top-3) ---')
for i, s in reranked:
    print(f'  청크#{i+1} (rerank={s:.3f}): {chunks[i][:60]}...')

**관찰:** Reranker 가 1차 결과의 순서를 바꾸거나, 1차에서 낮았던 청크가 위로 올라오기도 합니다. Cross-encoder 가 (쿼리, 청크) 관련성을 직접 평가했기 때문.

> 비용: 청크 10개 reranking 에 CPU 약 0.5-2초. GPU면 0.1초 이하. 정밀도가 중요한 서비스에선 거의 필수.


### C. 검색 품질 측정 — Recall@k

"하이브리드가 좋다더라" "Reranker 좋다더라" 같은 말은 *측정* 없이는 믿을 수 없습니다. 실서비스 챗봇 만들 때 첫 번째로 해야 할 것:

> **Golden set** — 질문과 그 질문의 "정답 청크" 인덱스 5~20쌍을 직접 만든다.

그리고 **Recall@k** — top-k 검색 결과 안에 정답 청크가 포함되는 비율 — 으로 다양한 설정을 비교합니다.


In [ ]:
# 우리 SAMPLE_DOC 에 대한 미니 golden set.
# 각 (질문, 정답 청크 0-based 인덱스).
# 실제 서비스라면 도메인 전문가가 30-200개 만드는 게 보통.
GOLDEN_SET = [
    ("연차는 며칠이야?",              [0]),  # 제15조
    ("연차 신청 기한 알려줘",         [0]),
    ("출장 신청은 얼마 전에 해?",     [1]),  # 제22조
    ("국내 출장 일비 얼마?",         [1]),
    ("재택근무 신청 어떻게 해?",      [2]),  # 제30조
    ("부모님 상 사일?",              [3]),  # 제35조 (경조사)
    ("교육비 지원 한도가 얼마?",      [4]),  # 제40조
    ("52시간 초과 가능한가?",         [5]),  # 제45조 (시간외)
    ("야근 수당 몇 배 받아?",         [5]),
]

def recall_at_k(retrieval_fn, golden, k: int = 3):
    """retrieval_fn(query, top_k) → [(chunk_idx, score), ...] 형식 가정."""
    hits = 0
    for q, gold_idxs in golden:
        retrieved = retrieval_fn(q, k)
        retrieved_idx = {i for i, _ in retrieved}
        if any(g in retrieved_idx for g in gold_idxs):
            hits += 1
    return hits / len(golden)

print(f'golden set 크기: {len(GOLDEN_SET)}개\n')

# 네 가지 방법 비교
configs = [
    ("Dense only",        lambda q, k: dense_search_simple(q, k)),
    ("BM25 only",         lambda q, k: bm25_search(q, k)),
    ("Hybrid (Dense+BM25)", lambda q, k: hybrid_search(q, k)),
    ("Hybrid + Rerank",   lambda q, k: [(i, s) for i, s in
                                        rerank(q, [idx for idx, _ in hybrid_search(q, 10)], top_k=k)]),
]

print(f'{"방법":<25} {"Recall@1":>10} {"Recall@3":>10} {"Recall@5":>10}')
print('-' * 60)
for name, fn in configs:
    r1 = recall_at_k(fn, GOLDEN_SET, k=1)
    r3 = recall_at_k(fn, GOLDEN_SET, k=3)
    r5 = recall_at_k(fn, GOLDEN_SET, k=5)
    print(f'{name:<25} {r1:>10.2%} {r3:>10.2%} {r5:>10.2%}')

**관찰 포인트:**

- **Recall@1** 은 가장 엄격 — 첫 결과가 정답이어야 함. 어렵습니다.
- **Recall@5** 까지 가면 대부분 방법이 잡지만, 그 안에서 *순위가 정확한지* 가 답변 품질을 결정.
- 보통 결과: `Dense ≈ BM25` < `Hybrid` ≤ `Hybrid + Rerank`. 도메인/쿼리 분포에 따라 다름.

**서비스화 체크리스트:**

- ☐ 도메인 전문가와 golden set 30개 이상 만들기
- ☐ 청크 크기 / overlap / 임베더 / reranker on/off 등 설정별 Recall@k 표
- ☐ 새 모델/방법 도입 전 같은 golden set 으로 *측정 후* 결정
- ☐ Recall 외에 *Faithfulness* (답변이 검색 결과에 진짜 근거 있나) 도 측정 — RAGAS / TruLens 같은 라이브러리 활용

이게 "느낌으로 튜닝" 과 "데이터로 튜닝" 의 차이입니다.


### 더 나아가려면

본 노트북에서는 다루지 않았지만 실서비스 챗봇에서 흔히 쓰는 기법들:

| 기법 | 핵심 |
|---|---|
| **Query expansion** (HyDE, multi-query) | 짧은 질문을 LLM 으로 풍부화해서 검색에 |
| **Contextual rewrite** | "그게 뭐였지?" 같은 대명사 질문을 self-contained 로 |
| **Vector DB** (pgvector / Qdrant / FAISS) | 청크 수가 늘면 numpy 한계, ANN 인덱스 필수 |
| **Agentic loop (function calling)** | 1회 검색 결과 부족 시 LLM 이 추가 검색 발행 |
| **Faithfulness check** | 답변이 검색 결과에 정말 근거 있는지 LLM 2차 패스 검증 |
| **A/B test 기록** | 실제 사용자 활동을 DB에 적재 → 회귀 측정 |

이 노트북은 챗봇의 *밑판* 까지였습니다. 위 기법들은 본 챕터 참고 프로젝트(Personal RAG: https://github.com/REAL-KENTECH/Personal-RAG)에 실제로 적용되어 있어 강의 후 코드 살펴보면 참고가 됩니다.


---
## 4-6. 본인 PDF 로 직접 실습

이제 위 흐름을 본인 PDF 한 부에 적용해 보겠습니다. 아래 순서:

1. PDF 파일 업로드 (Colab: 좌측 폴더 아이콘 → 업로드)
2. 파일 경로를 `PDF_PATH` 에 지정
3. 셀 실행 → 청킹·임베딩·검색이 자동으로 본인 문서에 적용됨
4. 마지막 셀에서 본인 문서에 대해 질문해보기


In [ ]:
# 본인 PDF 경로 (예: 업로드한 회사 매뉴얼 / 규정)
PDF_PATH = '/content/내문서.pdf'   # ← 본인 파일 경로로 수정

import os
if not os.path.exists(PDF_PATH):
    print(f'⚠ 파일이 없습니다: {PDF_PATH}')
    print('Colab 좌측 폴더 아이콘 → 업로드 후 경로를 정확히 적어주세요.')
    print('샘플 문서(SAMPLE_DOC)로 계속 진행하려면 다음 셀들은 건너뛰세요.')
else:
    from pypdf import PdfReader
    reader = PdfReader(PDF_PATH)
    own_text = "\n\n".join(page.extract_text() or "" for page in reader.pages)
    print(f'PDF 읽기 완료: {len(reader.pages)} 페이지, {len(own_text)} 자')
    print('--- 첫 300자 미리보기 ---')
    print(own_text[:300])

In [ ]:
# 본인 문서로 청킹 + 임베딩 새로 만들기
if 'own_text' in globals() and own_text:
    own_chunks = good_chunk(own_text, size=500)
    print(f'본인 문서 청크 개수: {len(own_chunks)}')

    own_embeddings = embedder.encode(
        own_chunks, convert_to_numpy=True, normalize_embeddings=True,
        show_progress_bar=False,
    )
    print(f'본인 임베딩 shape: {own_embeddings.shape}')

    # 기존 chunks / chunk_embeddings 를 본인 문서로 교체해서 ask_question 이 그대로 적용
    chunks = own_chunks
    chunk_embeddings = own_embeddings
    print('\n이제 ask_question() 이 본인 문서를 참조합니다.')
else:
    print('PDF 가 로드되지 않았습니다. 위 셀을 먼저 실행하거나 SAMPLE_DOC 으로 계속하세요.')

In [ ]:
# 본인 문서에 질문해보기
# 예: 매뉴얼이라면 "OO 절차 알려줘", 계약서라면 "위약금 조항은?" 등

YOUR_QUESTION = "이 문서의 핵심 내용을 5줄로 요약해줘"
print(ask_question(YOUR_QUESTION, top_k=5))

---
## 마무리 — 오늘 배운 것

✅ **단계 1: 청킹** — 의미 단위로 자르기. 단락 → 문장 경계 우선.
✅ **단계 2: 임베딩** — 텍스트를 384차원 벡터로. 비슷한 의미는 가까이.
✅ **단계 3: 검색** — 쿼리와 청크 임베딩의 코사인 유사도로 top-k.
✅ **단계 4: LLM 통합** — 검색 결과를 컨텍스트로 system prompt 와 함께 전달.
✅ **단계 5: 통합 함수** — 한 줄로 어떤 질문에든 답.

### 자주 발생하는 문제
| 증상 | 원인 | 해결 |
|---|---|---|
| 답변이 일반론적 | RAG 가 꺼져 있음 (just LLM) | `ask_question()` 처럼 검색 결과를 컨텍스트로 넣기 |
| 엉뚱한 청크가 검색됨 | 청킹이 의미 단위가 아님 | `good_chunk` 처럼 단락/문장 경계 사용 |
| 환각 발생 | system prompt 가 약함 | "Context 만 근거로", "없으면 모른다고" 강제 |
| PDF 텍스트 추출 안 됨 | 스캔 이미지 PDF | OCR (e.g. Tesseract) 또는 Docling 사용 |

### 다음 단계
- **Streamlit 으로 웹 UI 만들기** → 본 챕터 참고 프로젝트(Personal RAG) 코드 참고
- **Vector DB 도입** (pgvector / Qdrant / FAISS) — 문서가 많아지면
- **Agentic loop** — LLM 이 추가 검색을 자율적으로 결정
- **평가 프레임워크** — golden set + Recall@k / MRR / Faithfulness

> "오늘 만든 챗봇을 내일 바로 쓸 수 있습니다."
